# Apt 305, 50 Barry St — Original vs Modified pyBuildingEnergy Engine

This notebook compares the **annual heating and cooling energy need** for my apartment
(Apt 305, 50 Barry Street, Carlton, Melbourne) computed by two different versions of the
ISO 52016-1 engine:

1. **Original engine** — the official PyPI release of
   [`pybuildingenergy`](https://github.com/EURAC-EEBgroup/pyBuildingEnergy) (EURAC-EEBgroup).
   Same approach as my own simulation in
   `copy_of_pybuildingenergy_my_house.ipynb`: fixed heating/cooling setpoints (18 °C / 26 °C),
   PVGIS weather.

2. **Modified engine** — the
   [Sarthak790/pybuildinenergy_AIB](https://github.com/Sarthak790/pybuildinenergy_AIB) fork,
   as used in `pyBuildingEnergy/tests/sample_test.py`. The key engine-level changes in that
   fork are:
   - **Dynamic NCC 2022 setpoints**: heating/cooling setpoints are derived from the building's
     NCC climate zone (via `get_ncc_setpoints`) instead of being fixed by the user.
   - **Explicit schedule injection**: `Temperature_and_Energy_needs_calculation` accepts
     hourly occupant/appliance/lighting schedules directly as keyword arguments.
   - **Latent heat balance**: the engine additionally computes `Q_Latent` and `x_air_in`
     (moisture content) columns — a latent/humidity balance the original engine does not
     produce.
   - **DHW + infiltration**: the fork's test layers domestic hot water energy and an explicit
     infiltration ACH rate on top of the ISO 52016 result.

To keep this comparison **apples-to-apples**, both engine runs below use:
- the **same building parameters** (geometry, envelope, internal gains, occupancy/appliance/
  lighting schedules) taken from `copy_of_pybuildingenergy_my_house.ipynb`,
- the **same PVGIS weather source**.

The only deliberate difference left in is the **setpoint policy** — the modified engine's
defining feature — so the comparison shows what that change alone does to annual heating/
cooling need. Latent heat and DHW are *not* included here, to keep the comparison limited to
heating/cooling need (the same quantity in both engines).


## 1. Install the original engine (PyPI) and clone the modified engine (fork)

In [ ]:
# Original pyBuildingEnergy — official PyPI release
!pip install -q pybuildingenergy

# Modified engine — Sarthak790's AIB fork (cloned, not installed, so we can pick its
# source path explicitly when we want to use it)
import os
FORK_DIR = "/content/pybuildingenergy_AIB_fork"
if not os.path.exists(FORK_DIR):
    !git clone -q https://github.com/Sarthak790/pybuildinenergy_AIB.git {FORK_DIR}


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WEATHER_SOURCE = "pvgis"   # same weather source used for BOTH engines below


## 2. Building parameters (shared by both engines)

These are exactly the parameters from `copy_of_pybuildingenergy_my_house.ipynb`: a 20 m²
studio apartment with a west-facing brick facade, two small windows, five adjacent thermal
zones, and the occupancy/appliance/lighting schedules I derived for my own living pattern.

`build_bui()` returns a **fresh dictionary** every time it's called, so the original-engine
run and the modified-engine run never accidentally share (and mutate) the same object.

In [ ]:
def build_bui():
    """Returns a fresh copy of the Apt 305 building dictionary (no shared state)."""

    # --- Construction U-values & thermal capacity — Australian BCA 2006 minimum-spec ---
    U_EXT_WALL  = 1.00   # brick veneer / precast w/ R1.0 insulation
    U_INT_WALL  = 2.50   # concrete block + plasterboard, no insulation
    U_INT_SLAB  = 1.80   # 200 mm concrete intermediate floor
    U_WINDOW    = 5.40   # aluminium-frame single glazing
    G_WINDOW    = 0.65   # SHGC of clear single glazing

    ABS_EXT_WALL = 0.75  # dark red brick
    ABS_INT      = 0.0

    C_EXT_WALL = 450_000   # heavy concrete external wall, J/m2K
    C_INT_WALL = 330_000   # concrete-block partition
    C_INT_SLAB = 480_000   # 200 mm concrete slab
    C_WINDOW   = 0

    # --- Geometry ---
    LEN_NS  = 5.0   # N-S length, m (west facade width, along Barry St)
    LEN_EW  = 4.0   # E-W depth, m
    HEIGHT  = 2.7   # ceiling height, m

    FLOOR_AREA = LEN_NS * LEN_EW
    VOLUME     = FLOOR_AREA * HEIGHT

    A_WEST_GROSS  = LEN_NS * HEIGHT
    A_EAST_GROSS  = LEN_NS * HEIGHT
    A_NORTH_GROSS = LEN_EW * HEIGHT
    A_SOUTH_GROSS = LEN_EW * HEIGHT

    WIN_WIDTH_FIXED,    WIN_HEIGHT_FIXED    = 0.9, 0.9
    WIN_WIDTH_OPERABLE, WIN_HEIGHT_OPERABLE = 0.9, 0.9
    A_WINDOW_FIXED    = WIN_WIDTH_FIXED * WIN_HEIGHT_FIXED
    A_WINDOW_OPERABLE = WIN_WIDTH_OPERABLE * WIN_HEIGHT_OPERABLE
    A_WINDOW_TOTAL    = A_WINDOW_FIXED + A_WINDOW_OPERABLE
    A_WEST_OPAQUE     = A_WEST_GROSS - A_WINDOW_TOTAL

    bui = {
        "building": {
            "name": "Apt_305_50_Barry_St_Carlton",
            "azimuth_relative_to_true_north": 0,
            "latitude":  -37.800,
            "longitude": 144.968,
            "exposed_perimeter": 0,
            "height": HEIGHT,
            "wall_thickness": 0.20,
            "n_floors": 1,
            "building_type_class": "Residential_apartment",
            "adj_zones_present": True,
            "number_adj_zone": 5,
            "net_floor_area": FLOOR_AREA,
            "construction_class": "class_iii",
            "construction_year": "2006-today",
            "country": "Australia",
        },
        "adjacent_zones": [
            {
                "name": "apt_above",
                "orientation_zone": {"azimuth": 270.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_below",
                "orientation_zone": {"azimuth": 270.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_north",
                "orientation_zone": {"azimuth": 0.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_south",
                "orientation_zone": {"azimuth": 180.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "corridor",
                "orientation_zone": {"azimuth": 90.0},
                "area_facade_elements": np.array([81.0, 5.4, 81.0, 5.4, 60.0, 60.0]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL] * 6),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": 162.0,
                "building_type_class": "Residential_apartment",
                "a_use": 60.0,
            },
        ],
        "building_surface": [
            {
                "name": "West exterior wall (opaque)", "type": "opaque", "area": A_WEST_OPAQUE,
                "sky_view_factor": 0.5, "u_value": U_EXT_WALL, "solar_absorptance": ABS_EXT_WALL,
                "thermal_capacity": C_EXT_WALL, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": HEIGHT, "length": LEN_NS,
            },
            {
                "name": "North wall to Apt 306", "type": "opaque", "area": A_NORTH_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 0.0, "tilt": 90.0},
                "name_adj_zone": "apt_north", "height": HEIGHT, "length": LEN_EW,
            },
            {
                "name": "South wall to Apt 304", "type": "opaque", "area": A_SOUTH_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 180.0, "tilt": 90.0},
                "name_adj_zone": "apt_south", "height": HEIGHT, "length": LEN_EW,
            },
            {
                "name": "East wall to corridor", "type": "opaque", "area": A_EAST_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 90.0, "tilt": 90.0},
                "name_adj_zone": "corridor", "height": HEIGHT, "length": LEN_NS,
            },
            {
                "name": "Floor to Apt 205", "type": "opaque", "area": FLOOR_AREA,
                "sky_view_factor": 0.0, "u_value": U_INT_SLAB, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_SLAB, "orientation": {"azimuth": 0.0, "tilt": 0.0},
                "name_adj_zone": "apt_below", "height": LEN_NS, "length": LEN_EW,
            },
            {
                "name": "Ceiling to Apt 405", "type": "opaque", "area": FLOOR_AREA,
                "sky_view_factor": 0.0, "u_value": U_INT_SLAB, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_SLAB, "orientation": {"azimuth": 0.0, "tilt": 0.0},
                "name_adj_zone": "apt_above", "height": LEN_NS, "length": LEN_EW,
            },
            {
                "name": "West window — fixed", "type": "transparent", "area": A_WINDOW_FIXED,
                "sky_view_factor": 0.5, "u_value": U_WINDOW, "solar_absorptance": 0.5,
                "thermal_capacity": C_WINDOW, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": WIN_HEIGHT_FIXED, "g_value": G_WINDOW,
                "width": WIN_WIDTH_FIXED, "parapet": 1.0, "shading": True,
                "shading_type": "horizontal_overhang", "width_or_distance_of_shading_elements": 0.05,
                "overhang_proprieties": {"width_of_horizontal_overhangs": 0.25},
            },
            {
                "name": "West window — operable", "type": "transparent", "area": A_WINDOW_OPERABLE,
                "sky_view_factor": 0.5, "u_value": U_WINDOW, "solar_absorptance": 0.5,
                "thermal_capacity": C_WINDOW, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": WIN_HEIGHT_OPERABLE, "g_value": G_WINDOW,
                "width": WIN_WIDTH_OPERABLE, "parapet": 1.0, "shading": True,
                "shading_type": "horizontal_overhang", "width_or_distance_of_shading_elements": 0.05,
                "overhang_proprieties": {"width_of_horizontal_overhangs": 0.25},
            },
        ],
        "units": {
            "area": "m\u00b2", "u_value": "W/m\u00b2K", "thermal_capacity": "J/m\u00b2K",
            "azimuth": "degrees (0=N, 90=E, 180=S, 270=W)", "tilt": "degrees (0=horizontal, 90=vertical)",
            "internal_gain": "W/m\u00b2", "HVAC_profile": "0: off, 1: on",
        },
        "building_parameters": {
            "temperature_setpoints": {
                # Fixed setpoints, as used in copy_of_pybuildingenergy_my_house.ipynb.
                # The modified-engine cell overrides these with NCC zone-derived setpoints.
                "heating_setpoint": 18.0,
                "heating_setback":  15.0,
                "cooling_setpoint": 26.0,
                "cooling_setback":  28.0,
                "units": "\u00b0C",
            },
            "system_capacities": {
                "heating_capacity": 10_000_000.0,
                "cooling_capacity": 10_000_000.0,
                "units": "W",
            },
            "ventilation": {
                "ventilation_type": "occupancy",
                "flow_rate_per_person": 2.0,
                "units": "l/(s m\u00b2)",
                "custom_heat_transfer_coefficient_ventilation": None,
            },
            "internal_gains": [
                {
                    "name": "occupants", "full_load": 8.0,
                    "weekday": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.4,0.5,0.5,0.5,0.4,0.5,0.5,0.5,0.5,0.5,1.0,1.0,1.0,1.0],
                    "weekend": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,0.7,0.7,0.7,0.7,0.5,0.5,0.7,0.8,1.0,1.0,1.0,1.0,1.0,1.0],
                },
                {
                    "name": "appliances", "full_load": 25.0,
                    "weekday": [0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.3,0.4,1.0,0.6,0.4,0.2],
                    "weekend": [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.4,0.3,0.3,0.4,0.3,0.3,0.3,0.3,0.4,0.4,0.5,1.0,0.6,0.4,0.2],
                },
                {
                    "name": "lighting", "full_load": 3.0,
                    "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                    "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.3,0.3,0.2,0.2,0.2,0.2,0.2,0.2,0.3,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                },
            ],
            "construction": {
                "wall_thickness": 0.20,
                "thermal_bridges": 1.5,
                "units": "m (thickness), W/mK (thermal bridges)",
            },
            "climate_parameters": {"coldest_month": 7, "units": "1-12 (January-December)"},
            "heating_profile": {
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0],
            },
            "cooling_profile": {
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
            },
            "ventilation_profile": {
                "weekday": [1.0] * 24,
                "weekend": [1.0] * 24,
            },
        },
    }
    return bui

FLOOR_AREA = build_bui()["building"]["net_floor_area"]
print(f"Floor area: {FLOOR_AREA} m2")


## 3. Run the ORIGINAL engine (PyPI `pybuildingenergy`, fixed 18 °C / 26 °C setpoints)

In [ ]:
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI as orig_sanitize
from pybuildingenergy.source.utils import ISO52016 as OrigISO52016

bui_original = build_bui()
bui_original_fixed, report_original = orig_sanitize(bui_original, fix=True)

print("=== Validation (original engine) ===")
for r in report_original:
    fix_tag = " (fix applied)" if r["fix_applied"] else ""
    print(f"  [{r[\'level\']}] {r[\'path\']}: {r[\'msg\']}{fix_tag}")

errors_original = [r for r in report_original if r["level"] == "ERROR"]
if errors_original:
    raise ValueError("Building input invalid for the original engine — fix errors above.")


In [ ]:
print(f"=== Running ORIGINAL engine (weather: {WEATHER_SOURCE}) ===")
# The PyPI release returns a 2-tuple (hourly_sim, annual_results_df); the fork's engine
# (used below) returns a 3-tuple that also includes Sankey flow data. Handle both.
_orig_out = OrigISO52016.Temperature_and_Energy_needs_calculation(
    bui_original_fixed, weather_source=WEATHER_SOURCE
)
if len(_orig_out) == 3:
    hourly_original, annual_original, sankey_original = _orig_out
else:
    hourly_original, annual_original = _orig_out
    sankey_original = {}

Q_HC_original = hourly_original["Q_HC"]
heating_kWh_original = Q_HC_original[Q_HC_original > 0].sum() / 1000.0
cooling_kWh_original = -Q_HC_original[Q_HC_original < 0].sum() / 1000.0

print(f"\nORIGINAL engine — Heating need: {heating_kWh_original:8.1f} kWh/yr "
      f"({heating_kWh_original/FLOOR_AREA:5.1f} kWh/m2/yr)")
print(f"ORIGINAL engine — Cooling need: {cooling_kWh_original:8.1f} kWh/yr "
      f"({cooling_kWh_original/FLOOR_AREA:5.1f} kWh/m2/yr)")


## 4. Switch to the MODIFIED engine (AIB fork, NCC zone setpoints)

Both the PyPI package and the fork use the same top-level module name (`pybuildingenergy`),
so before importing the fork we clear any already-imported `pybuildingenergy.*` modules from
`sys.modules` and put the fork's `src/` directory **first** on `sys.path`. That way the next
`import` resolves to the fork's source, not the cached PyPI version.

In [ ]:
# Drop the PyPI package's cached modules so "pybuildingenergy" re-resolves to the fork
for mod_name in list(sys.modules):
    if mod_name == "pybuildingenergy" or mod_name.startswith("pybuildingenergy."):
        del sys.modules[mod_name]

FORK_SRC   = f"{FORK_DIR}/pyBuildingEnergy/src"
FORK_TESTS = f"{FORK_DIR}/pyBuildingEnergy/tests"
for p in (FORK_SRC, FORK_TESTS):
    if p not in sys.path:
        sys.path.insert(0, p)

from pybuildingenergy.source.check_input import sanitize_and_validate_BUI as mod_sanitize
from pybuildingenergy.source.utils import ISO52016 as ModISO52016
from climate_setpoints import get_ncc_setpoints, apply_setpoints_to_building

print("Modified engine imported from:", ModISO52016.__module__)


In [ ]:
bui_modified = build_bui()

# This is the core behavioural change in the fork: setpoints are derived from the
# building's NCC 2022 climate zone instead of being fixed by the user.
ncc_setpoints = get_ncc_setpoints(
    lat=bui_modified["building"]["latitude"],
    lon=bui_modified["building"]["longitude"],
)
bui_modified = apply_setpoints_to_building(bui_modified, ncc_setpoints)

print("=== NCC setpoints applied (modified engine) ===")
print(f"  NCC zone        : {ncc_setpoints[\'ncc_zone\']}")
print(f"  Heating setpoint: {ncc_setpoints[\'heating_setpoint\']} C")
print(f"  Cooling (bedroom): {ncc_setpoints[\'cooling_setpoint_bedroom\']} C")
print(f"  Cooling (living) : {ncc_setpoints[\'cooling_setpoint_living\']} C")
print(f"  Heating setback  : {ncc_setpoints[\'heating_setback\']} C")
print(f"  Cooling setback  : {ncc_setpoints[\'cooling_setback\']} C")


In [ ]:
bui_modified_fixed, report_modified = mod_sanitize(bui_modified, fix=True)

print("=== Validation (modified engine) ===")
for r in report_modified:
    fix_tag = " (fix applied)" if r["fix_applied"] else ""
    print(f"  [{r[\'level\']}] {r[\'path\']}: {r[\'msg\']}{fix_tag}")

errors_modified = [r for r in report_modified if r["level"] == "ERROR"]
if errors_modified:
    raise ValueError("Building input invalid for the modified engine — fix errors above.")


In [ ]:
print(f"=== Running MODIFIED engine (weather: {WEATHER_SOURCE}) ===")
_mod_out = ModISO52016.Temperature_and_Energy_needs_calculation(
    bui_modified_fixed, weather_source=WEATHER_SOURCE
)
if len(_mod_out) == 3:
    hourly_modified, annual_modified, sankey_modified = _mod_out
else:
    hourly_modified, annual_modified = _mod_out
    sankey_modified = {}

Q_HC_modified = hourly_modified["Q_HC"]
heating_kWh_modified = Q_HC_modified[Q_HC_modified > 0].sum() / 1000.0
cooling_kWh_modified = -Q_HC_modified[Q_HC_modified < 0].sum() / 1000.0

print(f"\nMODIFIED engine — Heating need: {heating_kWh_modified:8.1f} kWh/yr "
      f"({heating_kWh_modified/FLOOR_AREA:5.1f} kWh/m2/yr)")
print(f"MODIFIED engine — Cooling need: {cooling_kWh_modified:8.1f} kWh/yr "
      f"({cooling_kWh_modified/FLOOR_AREA:5.1f} kWh/m2/yr)")


## 5. Side-by-side comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        "engine": "Original pyBuildingEnergy (PyPI)",
        "setpoints": "Fixed 18C / 26C",
        "heating_kWh": round(heating_kWh_original, 1),
        "cooling_kWh": round(cooling_kWh_original, 1),
        "heating_kWh_m2": round(heating_kWh_original / FLOOR_AREA, 1),
        "cooling_kWh_m2": round(cooling_kWh_original / FLOOR_AREA, 1),
    },
    {
        "engine": "Modified AIB fork",
        "setpoints": f"NCC zone {ncc_setpoints[\'ncc_zone\']} "
                     f"({ncc_setpoints[\'heating_setpoint\']}C / {ncc_setpoints[\'cooling_setpoint_living\']}C)",
        "heating_kWh": round(heating_kWh_modified, 1),
        "cooling_kWh": round(cooling_kWh_modified, 1),
        "heating_kWh_m2": round(heating_kWh_modified / FLOOR_AREA, 1),
        "cooling_kWh_m2": round(cooling_kWh_modified / FLOOR_AREA, 1),
    },
])
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width / 2, comparison_df["heating_kWh"], width, label="Heating need")
ax.bar(x + width / 2, comparison_df["cooling_kWh"], width, label="Cooling need")

ax.set_xticks(x)
ax.set_xticklabels(comparison_df["engine"], rotation=10, ha="right")
ax.set_ylabel("Annual energy need (kWh/yr)")
ax.set_title("Apt 305, 50 Barry St, Carlton\nOriginal vs Modified ISO 52016 engine")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
OUTPUT_PATH = "/content/engine_comparison_apt305.csv"
comparison_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
